# Notebook 03 — Build RAG Knowledge Base

**Objective:** Build the FAISS vector index from confirmed Kepler KOIs.

This notebook runs **once on CPU** (no GPU needed). It produces three files saved to `models/rag/`:
- `faiss_index.bin` — searchable vector database of 2,743 confirmed planets
- `confirmed_planets.csv` — metadata for display (names, parameters)
- `scaler.pkl` — StandardScaler to normalise new query vectors at inference

These files are loaded by the Gradio web app (`app/app.py`).

**Source data:** `data/Dataset_Machine_Learning_Exoplanets_2024/q1_q17_dr25_sup_koi_*.csv`  
**Features used for similarity:** period, duration, depth, planet radius, stellar temperature, stellar radius

In [ ]:
# Section 1 — Install (run once, skip if already installed)
# !pip install faiss-cpu scikit-learn pandas numpy -q

In [ ]:
# Section 2 — Imports and config
import pandas as pd
import numpy as np
import faiss
import pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler

import glob

DATA_DIR  = Path('../data/Dataset_Machine_Learning_Exoplanets_2024')
OUT_DIR   = Path('../models/rag')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The 6 physical features used to measure similarity between KOIs
FEATURES = [
    'koi_period',    # orbital period in days
    'koi_duration',  # transit duration in hours
    'koi_depth',     # transit depth in ppm (brightness drop)
    'koi_prad',      # planet radius in Earth radii
    'koi_steff',     # stellar effective temperature in Kelvin
    'koi_srad',      # stellar radius in solar radii
]

# Columns to keep in the metadata CSV for display in the app
META_COLS = [
    'kepid', 'kepoi_name', 'kepler_name', 'koi_disposition',
    'koi_period', 'koi_duration', 'koi_depth',
    'koi_prad', 'koi_steff', 'koi_srad', 'koi_teq',
]

print('Config ready.')
print(f'Output directory: {OUT_DIR.resolve()}')

In [ ]:
# Section 3 — Load the supplemental KOI catalogue
# The file has comment lines starting with '#' — pandas skips them with comment='#'
koi_files = sorted(glob.glob(str(DATA_DIR / 'q1_q17_dr25_sup_koi_*.csv')))
assert len(koi_files) == 1, f'Expected 1 KOI file, found: {koi_files}'

df = pd.read_csv(koi_files[0], comment='#')

print(f'Loaded: {koi_files[0]}')
print(f'Total rows : {len(df):,}')
print(f'Columns    : {list(df.columns)}')

In [ ]:
# Section 4 — Filter to CONFIRMED only and clean
confirmed = df[df['koi_disposition'] == 'CONFIRMED'].copy()
print(f'CONFIRMED rows (before NaN drop): {len(confirmed):,}')

# Keep only the columns we need
available_meta = [c for c in META_COLS if c in confirmed.columns]
confirmed = confirmed[available_meta].copy()

# Drop any row that is missing a value in any of the 6 FAISS feature columns
before = len(confirmed)
confirmed = confirmed.dropna(subset=FEATURES).reset_index(drop=True)
after = len(confirmed)

print(f'Rows after dropping NaN in features: {after:,}  (dropped {before - after})')
print()
print('Feature stats:')
print(confirmed[FEATURES].describe().round(2))

In [ ]:
# Section 5 — Normalise features with StandardScaler
#
# Why StandardScaler?
# The 6 features are on very different scales:
#   koi_period: 1 – 500 days
#   koi_depth:  10 – 10,000 ppm
#   koi_steff:  3,000 – 7,000 K
# Without scaling, koi_steff would dominate the distance calculation.
# StandardScaler puts every feature on mean=0, std=1.

scaler = StandardScaler()
X = scaler.fit_transform(confirmed[FEATURES].values.astype(np.float32))

# L2-normalise each row so that inner product = cosine similarity
norms = np.linalg.norm(X, axis=1, keepdims=True)
X_norm = (X / (norms + 1e-8)).astype(np.float32)

print(f'Feature matrix shape : {X_norm.shape}')
print(f'Mean of norms (should be ~1): {np.linalg.norm(X_norm, axis=1).mean():.4f}')

In [ ]:
# Section 6 — Build FAISS index
#
# IndexFlatIP = exact inner product search.
# After L2 normalisation, inner product == cosine similarity.
# 'Flat' means brute-force — checks every vector. Fine for 2,743 rows.

d = X_norm.shape[1]  # 6 dimensions
index = faiss.IndexFlatIP(d)
index.add(X_norm)

print(f'FAISS index built')
print(f'  Vectors  : {index.ntotal:,}')
print(f'  Dimensions : {d}')
print(f'  Index type : IndexFlatIP (cosine similarity)')

In [ ]:
# Section 7 — Sanity check: retrieve 5 most similar planets to a known KOI
#
# We query the first confirmed KOI against itself.
# The top result should be itself (score=1.0), and the next 5 should be physically similar.

QUERY_IDX = 0  # change this to any row index to test different planets

query_vec = X_norm[QUERY_IDX:QUERY_IDX+1]   # shape (1, 6)
D, I = index.search(query_vec, k=6)         # k=6: top result is itself, we show next 5

query_row = confirmed.iloc[QUERY_IDX]
print(f'Query: {query_row["kepoi_name"]}  ({query_row.get("kepler_name", "—")})')
print(f'  period={query_row["koi_period"]:.2f}d  '
      f'depth={query_row["koi_depth"]:.0f}ppm  '
      f'prad={query_row["koi_prad"]:.2f}Re  '
      f'steff={query_row["koi_steff"]:.0f}K')
print()
print('Top 5 most similar confirmed planets:')
print(f'{"Rank":<5} {"KOI":<12} {"Kepler Name":<20} {"Score":<8} {"Period(d)":<12} {"Depth(ppm)":<12} {"Prad(Re)":<10}')
print('-' * 80)

for rank, (score, idx) in enumerate(zip(D[0][1:], I[0][1:]), 1):
    r = confirmed.iloc[idx]
    kname = str(r.get('kepler_name', '—')).strip() or '—'
    print(f'{rank:<5} {r["kepoi_name"]:<12} {kname:<20} {score:<8.4f} '
          f'{r["koi_period"]:<12.2f} {r["koi_depth"]:<12.0f} {r["koi_prad"]:<10.2f}')

In [ ]:
# Section 8 — Save artifacts

index_path    = OUT_DIR / 'faiss_index.bin'
metadata_path = OUT_DIR / 'confirmed_planets.csv'
scaler_path   = OUT_DIR / 'scaler.pkl'

faiss.write_index(index, str(index_path))

confirmed.to_csv(metadata_path, index=True)  # row index = FAISS vector position

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print('Saved:')
print(f'  {index_path}      ({index.ntotal} vectors)')
print(f'  {metadata_path}  ({len(confirmed)} rows)')
print(f'  {scaler_path}')
print()
print('RAG knowledge base ready. Next: run app/app.py')